In [4]:
import pandas as pd

# Load the dataset
data = pd.read_csv('spam.csv', encoding='latin-1')
data = data[['v1', 'v2']]  # Select relevant columns
data.columns = ['label', 'message']  # Rename columns

In [5]:
# Remove duplicates
data.drop_duplicates(inplace=True)
# Check for missing values
print(data.isnull().sum())

label      0
message    0
dtype: int64


In [7]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# Function to clean the text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = text.strip()  # Remove leading/trailing whitespace
    return text

# Apply the cleaning function
data['message'] = data['message'].apply(clean_text)

[nltk_data] Downloading package stopwords to C:\Users\Sanskruti
[nltk_data]     Ingle\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [8]:
from nltk.stem import PorterStemmer

# Initialize stemmer
stemmer = PorterStemmer()

# Function to stem words
def stem_words(text):
    return ' '.join([stemmer.stem(word) for word in text.split()])
# Apply stemming
data['message'] = data['message'].apply(stem_words)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer

vectorizer = CountVectorizer()

# Fit and transform the messages
X = vectorizer.fit_transform(data['message'])
y = data['label']

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the messages
X_tfidf = tfidf_vectorizer.fit_transform(data['message'])

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Initialize the model
model = MultinomialNB()

# Train the model
model.fit(X_train, y_train)

MultinomialNB()

In [12]:
# Make predictions
y_pred = model.predict(X_test)

# Print classification report
print(classification_report(y_test, y_pred))

# Print confusion matrix
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.94      1.00      0.97       889
        spam       1.00      0.63      0.77       145

    accuracy                           0.95      1034
   macro avg       0.97      0.81      0.87      1034
weighted avg       0.95      0.95      0.94      1034

[[889   0]
 [ 54  91]]


In [13]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
   'alpha': [0.1, 0.5, 1.0, 1.5, 2.0]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(MultinomialNB(), param_grid, cv=5)

# Fit the model
grid_search.fit(X_train, y_train)

# Best parameters
print("Best parameters:", grid_search.best_params_)

Best parameters: {'alpha': 0.1}


In [14]:
# Use the best model from grid search
best_model = grid_search.best_estimator_

# Make predictions with the best model
y_pred_best = best_model.predict(X_test)

# Print classification report for the best model
print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       889
        spam       0.93      0.89      0.91       145

    accuracy                           0.97      1034
   macro avg       0.96      0.94      0.95      1034
weighted avg       0.97      0.97      0.97      1034

